# Ensemble Layer
This notebook performs the ensemble layer task by using the code and models from the other three layers. 
To prevent overfitting, we use a simple Logistic Regression meta-classifier with L2 regularization trained on a subset of the synthetic dataset.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import re
import warnings
from scipy.sparse import hstack
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
print("Imports successful")

## 1. Load Synthetic Data
We load a sample of the synthetic data to train our meta-classifier. Using the synthetic data ensures we have balanced features across the layers.

In [ ]:
print("Loading global validation data...")
df = pd.read_csv('../global_val.csv')
# Sample to keep execution fast for NLI, or use full validation set if you have time
df_sample = df.sample(n=200, random_state=42).reset_index(drop=True)
print(f"Sampled {len(df_sample)} articles from the validation set for ensemble training.")


## 2. Layer 1 (Pattern Detection)
Load the models and define the prediction function.

In [ ]:
print("Loading Layer 1 artifacts...")
DIR1 = '../layer1_pattern'
vectorizer_l1 = joblib.load(os.path.join(DIR1, 'tfidf_vectorizer.pkl'))
model_l1 = joblib.load(os.path.join(DIR1, 'best_model.pkl'))
scaler_l1 = joblib.load(os.path.join(DIR1, 'scaler.pkl'))

def extract_stylometric_features(texts):
    features = pd.DataFrame()
    features['text_len'] = texts.apply(lambda x: len(str(x)))
    features['word_count'] = texts.apply(lambda x: len(str(x).split()))
    features['avg_word_len'] = features['text_len'] / (features['word_count'] + 1)
    features['exclamation_count'] = texts.apply(lambda x: str(x).count('!'))
    features['question_count'] = texts.apply(lambda x: str(x).count('?'))
    features['caps_ratio'] = texts.apply(lambda x: sum(1 for word in str(x).split() if word.isupper()) / (len(str(x).split()) + 1))
    return features.values

def get_layer1_prob_fake(text):
    text_tfidf = vectorizer_l1.transform([text])
    stylo_features = extract_stylometric_features(pd.Series([text]))
    stylo_scaled = scaler_l1.transform(stylo_features)
    text_combined = hstack([text_tfidf, stylo_scaled])
    
    # Layer 1 label 1 is FAKE, 0 is REAL
    if hasattr(model_l1, "predict_proba"):
        return float(model_l1.predict_proba(text_combined)[0][1])
    return 0.85 if model_l1.predict(text_combined)[0] == 1 else 0.15

## 3. Layer 2 (Semantic Analysis)
Load the SentenceTransformer model. We'll extract embeddings and compute similarity to FAKE and REAL centroids directly, to avoid needing the unsaved sklearn models from Layer 2. This mimics the layer 2 semantic search logic!

In [ ]:
print("Loading Layer 2 artifacts...")
model_l2 = SentenceTransformer('all-MiniLM-L6-v2')

# Let's extract the centroids from the true/fake datasets for zero-shot semantic matching
print("Computing Layer 2 centroids from a subset of true/fake data...")
base_path = '../layer1_pattern'
base_true = pd.read_csv('../global_train.csv')[lambda x: x['label'] == 0].sample(n=1000, random_state=42)
base_fake = pd.read_csv('../global_train.csv')[lambda x: x['label'] == 1].sample(n=1000, random_state=42)

base_true['combined'] = base_true['title'] + ' ' + base_true['text']
base_fake['combined'] = base_fake['title'] + ' ' + base_fake['text']

real_embeddings = model_l2.encode(base_true['combined'].tolist(), show_progress_bar=False)
fake_embeddings = model_l2.encode(base_fake['combined'].tolist(), show_progress_bar=False)

real_centroid = np.mean(real_embeddings, axis=0, keepdims=True)
fake_centroid = np.mean(fake_embeddings, axis=0, keepdims=True)

from sklearn.metrics.pairwise import cosine_similarity

def get_layer2_prob_fake(text):
    emb = model_l2.encode([text], show_progress_bar=False)
    sim_fake = cosine_similarity(emb, fake_centroid)[0][0]
    sim_real = cosine_similarity(emb, real_centroid)[0][0]
    
    # Softmax-like conversion to probability
    exp_fake = np.exp(sim_fake * 10)
    exp_real = np.exp(sim_real * 10)


## 4. Layer 3 (NLI Verification)
Load the NLI pipeline and known facts.

In [ ]:
print("Loading Layer 3 artifacts...")
nli_pipeline = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1
)

KNOWN_FACTS = [
    "Vaccines are safe and effective at preventing diseases.",
    "Climate change is caused by human activity and is scientifically proven.",
    "The Earth is approximately 4.5 billion years old.",
    "COVID-19 is caused by the SARS-CoV-2 coronavirus.",
    "Antibiotics do not work against viral infections.",
    "The United States has 50 states.",
    "World War II ended in 1945.",
    "The Moon orbits the Earth.",
    "Water boils at 100 degrees Celsius at standard atmospheric pressure.",
    "The human body has 206 bones.",
    "News articles that use excessive capital letters or emotional language are often unreliable.",
    "Credible news sources cite named experts and provide verifiable references."
]

def truncate_text(text, max_words=200):
    words = text.split()
    if len(words) > max_words:
        return " ".join(words[:max_words]) + "..."
    return text

def get_layer3_prob_fake(text):
    premise = truncate_text(str(text))
    contradiction_scores = []
    entailment_scores = []
    
    # Check top 3 facts to speed up execution
    for fact in KNOWN_FACTS[:3]:
        output = nli_pipeline(
            sequences=premise,
            candidate_labels=["entailment", "neutral", "contradiction"],
            hypothesis_template="{}"
        )
        score_map = dict(zip(output["labels"], output["scores"]))
        contradiction_scores.append(score_map.get("contradiction", 0))
        entailment_scores.append(score_map.get("entailment", 0))
        
    mean_contradiction = float(np.mean(contradiction_scores))
    mean_entailment = float(np.mean(entailment_scores))
    
    # If contradiction > entailment, it's more likely fake
    raw_conf = mean_contradiction + 0.5 * (mean_contradiction - mean_entailment)
    return float(np.clip(raw_conf, 0.01, 0.99))

## 5. Extract Features for Ensemble
We compute the probability of FAKE from each layer to use as features for the meta-classifier.

In [ ]:
print("Loading global validation data...")
df = pd.read_csv('../global_val.csv')
# Sample to keep execution fast for NLI, or use full validation set if you have time
df_sample = df.sample(n=200, random_state=42).reset_index(drop=True)
print(f"Sampled {len(df_sample)} articles from the validation set for ensemble training.")


## 6. Train Meta-Classifier (Logistic Regression)
To ensure the ensemble does not overfit, we use a simple Logistic Regression model with L2 regularization. A simple linear combination of probabilities is robust to overfitting.

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize Logistic Regression with L2 regularization
meta_model = LogisticRegression(penalty='l2', C=1.0, random_state=42)

# Perform 5-fold cross validation to ensure it doesn't overfit
cv_scores = cross_val_score(meta_model, X_train, y_train, cv=5)
print(f"Cross-Validation Accuracy: {cv_scores.mean()*100:.2f}% (+/- {cv_scores.std()*100:.2f}%)")

# Train on full training set
meta_model.fit(X_train, y_train)

# Evaluate on test set
y_pred = meta_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\nTest Set Accuracy: {acc*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["REAL", "FAKE"]))

# Display Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['REAL', 'FAKE'], yticklabels=['REAL', 'FAKE'])
plt.title('Ensemble Meta-Classifier Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Print coefficients to see how much weight each layer gets
print("\nEnsemble Weights (Layer Contributions):")
print(f"Layer 1 (Pattern): {meta_model.coef_[0][0]:.4f}")
print(f"Layer 2 (Semantic): {meta_model.coef_[0][1]:.4f}")
print(f"Layer 3 (NLI): {meta_model.coef_[0][2]:.4f}")

## 7. Final Prediction Function and Export
Wrap everything into a final ensemble predict function.

In [ ]:
# Save the meta-model
joblib.dump(meta_model, 'ensemble_meta_model.pkl')

def ensemble_predict(title, text):
    combined = str(title) + " " + str(text)
    
    l1_prob = get_layer1_prob_fake(text)
    l2_prob = get_layer2_prob_fake(combined)
    l3_prob = get_layer3_prob_fake(combined)
    
    features = np.array([[l1_prob, l2_prob, l3_prob]])
    
    # Predict
    final_prob_fake = meta_model.predict_proba(features)[0][1]
    
    label = "FAKE" if final_prob_fake > 0.5 else "REAL"
    confidence = final_prob_fake if label == "FAKE" else 1 - final_prob_fake
    
    return {
        "label": label,
        "confidence": round(float(confidence), 4),
        "layer_breakdown": {
            "layer1_fake_prob": round(float(l1_prob), 4),
            "layer2_fake_prob": round(float(l2_prob), 4),
            "layer3_fake_prob": round(float(l3_prob), 4)
        }
    }

# Test the ensemble
test_title = "Scientists confirm vaccines are safe and effective"
test_text = "Health authorities confirmed that the latest round of flu vaccinations has proven effective in reducing hospitalisation rates among vulnerable populations."
print("Testing Ensemble Prediction:")
print(ensemble_predict(test_title, test_text))